# Proyecto LSP — PUCP-305 (MediaPipe precalculado) en Colab

Entrena el reconocedor de **299 glosas médicas LSP** a partir del paquete de Zenodo
(record 13691887), que YA trae los landmarks de MediaPipe en pickles (no hay que correr MediaPipe).

**Objetivo del proyecto:** Top-5 accuracy ≥ 90% (el top-1 a 299 clases con ~17
muestras/clase es nivel investigación; el Top-5 alimenta luego un LLM que arma la frase).

**Flujo:**
1. Clonar repo, instalar `mediapipe`+`tensorflow`, montar Drive.
2. Descargar `ANNOTATIONS.zip` y `MEDIAPIPE.zip` de Zenodo DIRECTO a Colab (`/content`).
3. Limpiar anotaciones: quitar clase 222 (`#N/A`) y remapear a 0..298 → **299 clases**.
4. Diagnóstico: longitud de secuencias + tasa de detección de manos.
5. Extraer cada `.pkl` → tensor `(FRAMES, 258)` → augmentation → normalización.
6. BiLSTM + class_weights con métrica Top-5.
7. Evaluar **top-1 y top-5** y guardar modelo + clases en Drive.

**Pre-requisito:** GPU T4 activada. Los datos se bajan solos de Zenodo (no necesitas Drive para ellos).

## Setup — repo, dependencias y Drive

In [ ]:
%cd /content
!rm -rf /content/Proyecto_Percepcion
!git clone -q https://github.com/Jtarazona00/Proyecto_Percepcion.git /content/Proyecto_Percepcion
%cd /content/Proyecto_Percepcion

import os, sys
os.environ['DATASET'] = 'pucp305'  # activar ANTES de importar config
sys.path.insert(0, '/content/Proyecto_Percepcion')

!pip install -q tensorflow==2.18.0 mediapipe==0.10.14

import tensorflow as tf
from google.colab import drive
drive.mount('/content/drive')
print('GPU:', tf.config.list_physical_devices('GPU'))
print('TF :', tf.__version__)

## Paso 1 — Descargar datos de Zenodo (directo a Colab)

Descarga desde el record de Zenodo al disco temporal de Colab (`/content`, ~100 GB libres).
**NO usa Drive para los datos de entrada** (así evitas el límite de almacenamiento de Drive).
`MEDIAPIPE.zip` son 3.5 GB; `wget -c` reanuda si la descarga se corta.
En Drive solo se guardan las salidas (modelo + caché de arrays, ~250 MB).

In [ ]:
from pathlib import Path

# Salidas en Drive (ligero). Datos de entrada en el disco temporal de Colab.
MODELS_DRIVE = Path('/content/drive/MyDrive/PUCP305_models')
CACHE_DRIVE  = MODELS_DRIVE / 'cache'   # arrays .npy extraidos (para no re-extraer)
for d in (MODELS_DRIVE, CACHE_DRIVE):
    d.mkdir(parents=True, exist_ok=True)

DATA = Path('/content/pucp305'); DATA.mkdir(exist_ok=True)
MEDIAPIPE_DIR = Path('/content/MEDIAPIPE')
ZEN = 'https://zenodo.org/records/13691887/files'

ANN_ZIP = DATA / 'ANNOTATIONS.zip'
REF_CSV = DATA / 'videos_ref_annotations.csv'
MP_ZIP  = DATA / 'MEDIAPIPE.zip'

# Archivos chicos (KB)
!wget -c -q -O {str(ANN_ZIP)} '{ZEN}/ANNOTATIONS.zip?download=1'
!wget -c -q -O {str(REF_CSV)} '{ZEN}/videos_ref_annotations.csv?download=1'
# Archivo grande (3.5 GB) con barra de progreso y reanudacion
!wget -c -q --show-progress -O {str(MP_ZIP)} '{ZEN}/MEDIAPIPE.zip?download=1'

import os
assert os.path.getsize(MP_ZIP) > 3.0e9, 'MEDIAPIPE.zip incompleto; re-ejecuta esta celda (wget -c reanuda).'

# Descomprimir MediaPipe una sola vez -> /content/MEDIAPIPE/<FILENAME>.pkl
if not MEDIAPIPE_DIR.exists() or len(list(MEDIAPIPE_DIR.glob('*.pkl'))) < 7000:
    !unzip -n -q {str(MP_ZIP)} -d /content
n_pkl = len(list(MEDIAPIPE_DIR.glob('*.pkl')))
print(f'pickles disponibles: {n_pkl}')

## Paso 2 — Limpiar anotaciones (299 clases)

Quita la clase 222 (`#N/A`, 15 muestras) y remapea los IDs a `0..298` contiguos.
Las variantes de señas (`ACCIDENTE2`, compuestas `^`, sufijos `(..)`) se mantienen.

In [ ]:
import config
from src.preprocesamiento.extraer_pkl_pucp305 import limpiar_anotaciones

splits, labels_map = limpiar_anotaciones(ANN_ZIP, REF_CSV)
config.set_classes(labels_map.sort_values('NEW_ID').LABEL.tolist())

print(f'Clases activas: {config.NUM_CLASSES}  |  FRAMES = {config.FRAMES}')
for s, df in splits.items():
    print(f'  {s:6} n={len(df):5} clases={df.CLASS_ID.nunique()}')
labels_map.head()

## Paso 3 — Diagnóstico

Antes de extraer todo, mide la longitud de las secuencias (para validar `FRAMES`)
y la **tasa de detección de manos**. Si la Holistic legacy detecta pocas manos, el
fallback a la API Tasks debería subir bastante ese %.

In [ ]:
from src.preprocesamiento.extraer_pkl_pucp305 import diagnostico
_ = diagnostico(splits['train'], MEDIAPIPE_DIR, n_muestra=300)

## Paso 4 — Extraer features `(N, FRAMES, 258)`

Cada `.pkl` (lista de frames) → remuestreo a `config.FRAMES` frames → vector de 258
por frame (pose 132 + mano izq 63 + mano der 63). El caché se versiona por FRAMES,
así que si cambias `config.FRAMES` se vuelve a extraer automáticamente.

In [ ]:
import numpy as np
from src.preprocesamiento.extraer_pkl_pucp305 import construir_arrays

def cargar_o_extraer(nombre, df):
    suf = f'{nombre}_f{config.FRAMES}'
    fx, fy = CACHE_DRIVE / f'X_{suf}.npy', CACHE_DRIVE / f'y_{suf}.npy'
    if fx.exists() and fy.exists():
        print(f'{nombre}: cargado de cache ({fx.name})')
        return np.load(fx), np.load(fy)
    print(f'{nombre}: extrayendo (FRAMES={config.FRAMES})...')
    X, y, faltantes = construir_arrays(df, MEDIAPIPE_DIR, frames_objetivo=config.FRAMES)
    np.save(fx, X); np.save(fy, y)
    if faltantes:
        print(f'  AVISO: {len(faltantes)} pkl faltantes/invalidos (ej: {faltantes[:3]})')
    return X, y

X_train, y_train = cargar_o_extraer('train', splits['train'])
X_val,   y_val   = cargar_o_extraer('val',   splits['val'])
X_test,  y_test  = cargar_o_extraer('test',  splits['test'])
print(f'\nTrain {X_train.shape}  Val {X_val.shape}  Test {X_test.shape}')

## Paso 5 — Augmentation + Normalización

1. **Augmentation** (`src/utils/augmentacion.py`): ruido, escala, traslación,
   frame-dropout y mirror (intercambia manos). Clave con ~17 muestras/clase.
2. **Normalización** (`src/utils/normalizacion.py`): centra cada frame en el punto
   medio de los hombros y escala por el ancho de hombros → invarianza a posición y
   tamaño de la persona. Se aplica DESPUÉS de augmentar (el mirror asume coords [0,1]).
   Se generan copias `_n`; los arrays crudos del caché quedan intactos.

In [ ]:
from src.utils.augmentacion import expandir_train_set
from src.utils.normalizacion import normalizar_batch

X_train_aug, y_train_aug = expandir_train_set(X_train, y_train, factor=4)
print(f'Train original:  {X_train.shape}')
print(f'Train aumentado: {X_train_aug.shape}')

X_train_n = normalizar_batch(X_train_aug)
X_val_n   = normalizar_batch(X_val)
X_test_n  = normalizar_batch(X_test)
print('Normalizado: centro = hombros, escala = ancho de hombros.')

## Paso 6 — BiLSTM con métrica Top-5

Mismo BiLSTM (`src/modelos/lstm_modelo.py`) con salida de 299 clases. Se compila
con accuracy (top-1) **y** `SparseTopKCategoricalAccuracy(k=5)`; el checkpoint
guarda el mejor `val_top5`.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from src.modelos.lstm_modelo import construir_modelo

model = construir_modelo(num_classes=config.NUM_CLASSES)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=config.LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name='top5')],
)
model.summary()

ckpt = str(MODELS_DRIVE / 'modelo_pucp305_best.keras')
callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True),
    ModelCheckpoint(ckpt, monitor='val_top5', mode='max', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6),
]

clases = np.unique(y_train_aug)
pesos = compute_class_weight('balanced', classes=clases, y=y_train_aug)
class_weight = dict(zip(clases.tolist(), pesos.tolist()))

historial = model.fit(
    X_train_n, y_train_aug,
    validation_data=(X_val_n, y_val),
    epochs=config.EPOCHS, batch_size=64,
    callbacks=callbacks, class_weight=class_weight, verbose=1,
)

## Paso 7 — Evaluación (Top-1 y Top-5)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Curvas
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(historial.history['loss'], label='train'); ax[0].plot(historial.history['val_loss'], label='val'); ax[0].set_title('loss'); ax[0].legend()
ax[1].plot(historial.history['accuracy'], label='train'); ax[1].plot(historial.history['val_accuracy'], label='val'); ax[1].set_title('top-1'); ax[1].legend()
ax[2].plot(historial.history['top5'], label='train'); ax[2].plot(historial.history['val_top5'], label='val'); ax[2].set_title('top-5'); ax[2].legend()
plt.savefig('curvas_pucp305.png', dpi=120, bbox_inches='tight'); plt.show()

# Metricas en test (desde las predicciones; robusto a la version de Keras)
y_prob = model.predict(X_test_n, verbose=0)
y_pred = y_prob.argmax(1)
top1 = float((y_pred == y_test).mean())
top5_idx = np.argsort(y_prob, axis=1)[:, -5:]
top5 = float(np.mean([yt in fila for yt, fila in zip(y_test, top5_idx)]))
print(f"\nTest top-1: {top1:.2%}")
print(f"Test top-5: {top5:.2%}  <-- objetivo >= 90%")

from sklearn.metrics import classification_report
print('Macro F1:', classification_report(y_test, y_pred, output_dict=True, zero_division=0)['macro avg']['f1-score'])

## Paso 8 — Guardar modelo y clases en Drive

In [ ]:
import json, shutil

model.save(MODELS_DRIVE / 'modelo_pucp305_final.keras')
shutil.copy('curvas_pucp305.png', MODELS_DRIVE / 'curvas_pucp305.png')
labels_map.to_csv(MODELS_DRIVE / 'labels_map_pucp305.csv', index=False)
with open(MODELS_DRIVE / 'classes_pucp305.json', 'w', encoding='utf-8') as f:
    json.dump(config.CLASSES, f, ensure_ascii=False, indent=2)
print('Guardado en', MODELS_DRIVE)